# 05 · Mini-batches: slicing, generators, iterators, `Dataset` / `DataLoader`

The homework said: *"create three iterable objects … at each step the iterators must return two tensors"* and explicitly allowed either a custom generator/iterator or PyTorch's `Dataset` + `DataLoader`. The coding test may ask for either. You should be able to write **all** of them without help, and — more importantly — know exactly which **shapes and dtypes** come out.

---
## Theory in 60 seconds

- **Epoch** = one pass over the whole training set. **Mini-batch** = a small group of samples used for one parameter update.
- Why batches: one gradient from the full dataset is precise but expensive and gives one update per epoch; batches give many cheaper, noisier updates per epoch — and the noise even helps generalisation.
- Training batches should be drawn in a **new random order every epoch**. Evaluation order does not influence the metrics, so the test loader is not shuffled (and the homework required it to keep the order).
- An **iterable** has `__iter__()` that returns an **iterator**; an iterator has `__next__()` that returns the next item or raises `StopIteration`. A `for` loop calls `iter()` once and then `next()` repeatedly.
- A **generator function** (contains `yield`) is the short way to write an iterator. **A generator object can be consumed only once.**
- `Dataset`: knows `__len__()` and `__getitem__(idx)` for **one sample**. `DataLoader`: asks the dataset for `batch_size` samples and **stacks** them along a new first axis.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from dnd_utils import check, load_houses

d = load_houses()                      # the result of notebook 04, ready-made
X_train, y_train = d["X_train"], d["y_train"]          # NumPy float32: (720, 13) and (720,)
X_val, y_val = d["X_val"], d["y_val"]
X_test, y_test = d["X_test"], d["y_test"]
print(X_train.shape, y_train.shape, X_train.dtype)

---
## Exercise 5.1 · Batch arithmetic

720 training samples, batch size 32. Predict (no code): how many batches per epoch, and how large is the last one? Then write the general formulas for any `n` and `batch_size`.

In [ ]:
n_batches_720_32 = None
last_batch_size_720_32 = None

def n_batches(n, batch_size):
    raise NotImplementedError

def last_batch_size(n, batch_size):
    raise NotImplementedError

In [ ]:
check("23 batches", lambda: n_batches_720_32 == 23)
check("last batch has 16 samples", lambda: last_batch_size_720_32 == 16)
check("n_batches general", lambda: [n_batches(n, b) for n, b in ((720, 32), (64, 32), (65, 32), (5, 32))] == [23, 2, 3, 1])
check("last_batch_size general", lambda: [last_batch_size(n, b) for n, b in ((720, 32), (64, 32), (65, 32), (5, 32))] == [16, 32, 1, 5])

---
## Exercise 5.2 · Option 1: slicing inside a loop

Convert the training arrays to tensors (`Xt`: `(720, 13)`, `yt`: `(720,)`, both `float32`). Then loop over the data in consecutive batches of 32 and append the **shape pair** `(tuple(xb.shape), tuple(yb.shape))` of each batch to `shapes`. The label batch must have shape `(batch, 1)`.

Does slicing past the end (`Xt[704:736]`) raise an error?

In [ ]:
Xt = None
yt = None
shapes = []

In [ ]:
check("23 batches", lambda: len(shapes) == 23)
check("full batch shapes", lambda: shapes[0] == ((32, 13), (32, 1)))
check("last batch shapes", lambda: shapes[-1] == ((16, 13), (16, 1)))

---
## Exercise 5.3 · Option 2: a generator

Write `minibatch_generator(X, y, batch_size, shuffle=False, rng=None)`:

- `X`: tensor `(n, d)`, `y`: tensor `(n,)`
- yields `(xb, yb)` with `yb` of shape `(batch, 1)`
- if `shuffle` is true, the samples are visited in a random order drawn from `rng` (a `np.random.Generator`), a **new** order for every call; inputs and labels must stay paired.

<details><summary>Hint</summary>

Draw `perm = rng.permutation(n)` once at the top, then take `idx = perm[offset:offset + batch_size]` and index both tensors with `idx`. For the unshuffled case you can use `perm = np.arange(n)` and keep the same code path.
</details>

In [ ]:
def minibatch_generator(X, y, batch_size, shuffle=False, rng=None):
    raise NotImplementedError
    yield

In [ ]:
def _collect(**kw):
    xs, ys = zip(*minibatch_generator(Xt, yt, 32, **kw))
    return xs, ys
check("23 batches, shapes ok", lambda: len(_collect()[0]) == 23 and _collect()[0][0].shape == (32, 13) and _collect()[1][0].shape == (32, 1) and _collect()[1][-1].shape == (16, 1))
check("unshuffled: original order", lambda: torch.equal(torch.cat(_collect()[1])[:, 0], yt))
_rng = np.random.default_rng(0)
_xs, _ys = _collect(shuffle=True, rng=_rng)
check("shuffled: a different order ...", lambda: not torch.equal(torch.cat(_ys)[:, 0], yt))
check("... but every sample exactly once", lambda: torch.equal(torch.sort(torch.cat(_ys)[:, 0]).values, torch.sort(yt).values))
check("... and inputs still paired with labels", lambda: (lambda X_, y_: all(any(torch.equal(Xt[j], X_[i]) for j in (yt == y_[i]).nonzero()[:, 0]) for i in range(40)))(torch.cat(_xs), torch.cat(_ys)[:, 0]))
_ys2 = _collect(shuffle=True, rng=_rng)[1]
check("second epoch: yet another order", lambda: not torch.equal(torch.cat(_ys2), torch.cat(_ys)))

---
## Exercise 5.4 · The one-epoch trap

Predict the output of this cell before running it.

In [ ]:
gen = minibatch_generator(Xt, yt, 32)        # created ONCE, outside the epoch loop
for epoch in range(3):
    n_batches_seen = 0
    for xb, yb in gen:
        n_batches_seen += 1
    print(f"epoch {epoch}: {n_batches_seen} batches")

What happened, what would the loss curve of such a training look like, and how do you fix it?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 5.5 · Option 3: iterator and iterable classes

**a)** Write the explicit iterator `MinibatchIterator` (state: the current offset; `__next__` returns the next batch or raises `StopIteration`). No shuffling needed.

**b)** Write the iterable `BatchLoader`, which can be looped over **many times**: `__iter__` returns a fresh iterator each time (shuffled if requested), `__len__` returns the number of batches.

In [ ]:
class MinibatchIterator:
    def __init__(self, X, y, batch_size):
        raise NotImplementedError

    def __iter__(self):
        raise NotImplementedError

    def __next__(self):
        raise NotImplementedError


class BatchLoader:
    def __init__(self, X, y, batch_size, shuffle=False, seed=0):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

    def __iter__(self):
        raise NotImplementedError

In [ ]:
check("MinibatchIterator: 23 batches with the right shapes", lambda: [yb.shape[0] for _, yb in MinibatchIterator(Xt, yt, 32)] == [32] * 22 + [16])
check("MinibatchIterator: next() works step by step", lambda: (lambda it: next(it)[0].shape == (100, 13) and torch.equal(next(it)[1][:, 0], yt[100:200]))(MinibatchIterator(Xt, yt, 100)))
check("MinibatchIterator is exhausted after one pass", lambda: (lambda it: (len(list(it)), len(list(it))))(MinibatchIterator(Xt, yt, 32)) == (23, 0))
_bl = None
try: _bl = BatchLoader(Xt, yt, 32, shuffle=True)
except NotImplementedError: pass
check("BatchLoader: len", lambda: len(_bl) == 23)
check("BatchLoader: can be iterated in every epoch", lambda: [len(list(_bl)) for _ in range(3)] == [23, 23, 23])
check("BatchLoader: a new order each epoch", lambda: not torch.equal(torch.cat([y for _, y in _bl]), torch.cat([y for _, y in _bl])))

---
## Exercise 5.6 · Option 4: `Dataset` + `DataLoader`

Write `RegressionDataset`: the constructor receives NumPy arrays `x` `(n, d)` and `y` `(n,)` and stores them as `float32` tensors; `__getitem__(idx)` returns the pair for **one** sample such that a `DataLoader` yields batches of shape `(batch, d)` and `(batch, 1)`.

Then create `loader_train`, `loader_val` (both shuffled) and `loader_test` (not shuffled), batch size 32.

In [ ]:
class RegressionDataset(Dataset):
    def __init__(self, x, y):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

    def __getitem__(self, idx):
        raise NotImplementedError

loader_train = None
loader_val = None
loader_test = None

In [ ]:
_ds = None
try: _ds = RegressionDataset(X_train, y_train)
except NotImplementedError: pass
check("len(dataset) == 720", lambda: len(_ds) == 720)
check("one sample: shapes (13,) and (1,), float32", lambda: _ds[0][0].shape == (13,) and _ds[0][1].shape == (1,) and _ds[0][0].dtype == _ds[0][1].dtype == torch.float32)
check("train batch shapes (32, 13) and (32, 1)", lambda: (lambda b: b[0].shape == (32, 13) and b[1].shape == (32, 1))(next(iter(loader_train))))
check("len(loader_train) == 23 batches", lambda: len(loader_train) == 23)
check("train loader shuffles", lambda: not torch.equal(torch.cat([y for _, y in loader_train])[:, 0], torch.tensor(y_train)))
check("test loader keeps the order", lambda: torch.equal(torch.cat([y for _, y in loader_test])[:, 0], torch.tensor(y_test)))

---
## Exercise 5.7 · What does the `DataLoader` stack? Predict the batch shapes

The `DataLoader` takes `batch_size` results of `__getitem__` and stacks each element along a **new first axis**. So the label shape of a batch is decided entirely by what `__getitem__` returns for one sample.

For each variant predict the shape of the label batch (batch size 32) and say which loss it suits.

In [ ]:
class Variant(Dataset):
    """ys_shape: how the labels are stored; index: how __getitem__ indexes them."""
    def __init__(self, ys, index):
        self.ys, self.index = ys, index
    def __len__(self):
        return self.ys.shape[0]
    def __getitem__(self, idx):
        return self.index(self.ys, idx)

ys_1d = torch.tensor(y_train)                  # (720,)
ys_2d = torch.tensor(y_train)[:, None]         # (720, 1)
variants = {
    "A: ys (n,)   -> ys[idx]":        Variant(ys_1d, lambda ys, i: ys[i]),
    "B: ys (n,)   -> ys[idx, None]":  Variant(ys_1d, lambda ys, i: ys[i, None]),
    "C: ys (n, 1) -> ys[idx]":        Variant(ys_2d, lambda ys, i: ys[i]),
    "D: ys (n, 1) -> ys[idx, None]":  Variant(ys_2d, lambda ys, i: ys[i, None]),
}

In [ ]:
pred_shapes = {"A": None, "B": None, "C": None, "D": None}

In [ ]:
for name, ds in variants.items():
    actual = tuple(next(iter(DataLoader(ds, batch_size=32))).shape)
    check(f"{name:32s} -> {actual}", lambda: tuple(pred_shapes[name[0]]) == actual)

---
## Exercise 5.8 · The average loss of an epoch — exactly

The lecturer's notebook notes that averaging the per-batch losses is *"not exactly precise because of the last (possibly) incomplete batch"*. Make that concrete.

For the fixed (untrained) model below and `loader_test` (240 samples, **batch size 50** here, so the last batch has 40), compute:

- `mean_of_batch_means`: the plain average of the per-batch MSE values,
- `weighted_mean`: each batch loss weighted by its number of samples, divided by the total number of samples,
- `full_loss`: the MSE computed on the whole test set in one go.

Which two agree, and why?

In [ ]:
torch.manual_seed(0)
model = nn.Linear(13, 1)
loss_fn = nn.MSELoss()
loader50 = DataLoader(RegressionDataset(X_test, y_test), batch_size=50, shuffle=False)

mean_of_batch_means = None
weighted_mean = None
full_loss = None

In [ ]:
check("weighted mean == loss on the full set", lambda: abs(weighted_mean - full_loss) / full_loss < 1e-5)
check("plain mean of batch means is (slightly) different", lambda: abs(mean_of_batch_means - full_loss) / full_loss > 1e-5 and abs(mean_of_batch_means - full_loss) / full_loss < 0.2)

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 5.9 · A classification dataset

For multi-class classification the labels must come out of the loader as shape `(batch,)` and dtype `int64`. Write `ClassificationDataset` accordingly and build a shuffled loader with batch size 16 for the toy data below.

In [ ]:
rng = np.random.default_rng(1)
X_toy = rng.normal(size=(100, 4)).astype(np.float32)
y_toy = rng.integers(0, 3, size=100)                 # NumPy int64, values 0/1/2

class ClassificationDataset(Dataset):
    pass

loader_toy = None

In [ ]:
check("batch shapes (16, 4) and (16,)", lambda: (lambda b: b[0].shape == (16, 4) and b[1].shape == (16,))(next(iter(loader_toy))))
check("dtypes float32 and int64", lambda: (lambda b: b[0].dtype == torch.float32 and b[1].dtype == torch.int64)(next(iter(loader_toy))))
check("7 batches, last one has 4 samples", lambda: [len(y) for _, y in loader_toy] == [16] * 6 + [4])
check("usable with CrossEntropyLoss", lambda: nn.CrossEntropyLoss()(nn.Linear(4, 3)(next(iter(loader_toy))[0]), next(iter(loader_toy))[1]).ndim == 0)

---
## Blank-page drill

Cover the notebook. In the empty cell write from memory, for the arrays `X_train, y_train, X_val, y_val, X_test, y_test` (regression):

1. a `Dataset` subclass,
2. three `DataLoader`s with the correct `shuffle` settings,
3. a sanity check that prints the shape and dtype of one batch,
4. **and** the generator version `minibatch_generator` (with shuffling).

Self-check: `(32, 13) float32` and `(32, 1) float32`.

In [ ]:
# your code